In [ ]:
!pip install opendatasets --quiet
import opendatasets as od
od.download("https://www.kaggle.com/datasets/andrewmvd/animal-faces")

In [ ]:
import torch
from sklearn.preprocessing import LabelEncoder # Label Encoder to encode the classes from strings to numbers
import torch  # Main PyTorch Library
import torchvision.transforms as transforms  # Transform function used to modify and preprocess all the images
from torch.utils.data import Dataset, DataLoader # Dataset class and DataLoader for creating the objects
from PIL import Image # Used to read the images from the directory
from torch import nn
from torchsummary import summary
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu" # detect the GPU if any, if not use CPU, change cuda to mps if you have a mac
print("Device available: ", device)

In [ ]:
image_path = [] # Empty array where we will fill the paths of the images
labels = []

for i in os.listdir('/content/animal-faces/afhq/'):
  for label in os.listdir(f'/content/animal-faces/afhq/{i}/'):
    for image in os.listdir(f'/content/animal-faces/afhq/{i}/{label}/'):
      labels.append(label)
      image_path.append(f'/content/animal-faces/afhq/{i}/{label}/{image}')

In [ ]:
zip(image_path, labels)
data_df = pd.DataFrame(zip(image_path, labels), columns=["image_paths", "labels"])
data_df.head()

In [ ]:
	image_paths	                                        labels
0	/content/animal-faces/afhq/train/wild/flickr_w...	wild
1	/content/animal-faces/afhq/train/wild/flickr_w...	wild
2	/content/animal-faces/afhq/train/wild/flickr_w...	wild
3	/content/animal-faces/afhq/train/wild/flickr_w...	wild
4	/content/animal-faces/afhq/train/wild/flickr_w...	wild


In [ ]:
train = data_df.sample(frac=0.7, random_state=1)
test = data_df.drop(train.index)

val = test.sample(frac=0.5, random_state=1)
test = test.drop(val.index)

In [ ]:
print(train)   
        
                                            image_paths labels
15582  /content/animal-faces/afhq/val/dog/pixabay_dog...    dog
5816   /content/animal-faces/afhq/train/dog/pixabay_d...    dog
5657   /content/animal-faces/afhq/train/dog/pixabay_d...    dog
7400   /content/animal-faces/afhq/train/dog/flickr_do...    dog
4097   /content/animal-faces/afhq/train/wild/flickr_w...   wild
...                                                  ...    ...
3622   /content/animal-faces/afhq/train/wild/flickr_w...   wild
14673  /content/animal-faces/afhq/val/wild/pixabay_wi...   wild
6021   /content/animal-faces/afhq/train/dog/pixabay_d...    dog
12669  /content/animal-faces/afhq/train/cat/pixabay_c...    cat
15396  /content/animal-faces/afhq/val/dog/pixabay_dog...    dog

[11291 rows x 2 columns]

In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(data_df['labels'])

In [ ]:
data_transforms = transforms.Compose([
    transforms.Resize((128, 128)), # Resize images to 128x128 as per the model summary input size
    transforms.ToTensor(),
    transforms.ConvertImageDtype(torch.float)
])

Custom Dataset Class

In [ ]:
class CustomImageDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform
        self.labels = torch.tensor(label_encoder.transform(dataframe['labels'])).to(device)

    def __len__(self):
        return self.dataframe.shape[0]

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx, 0]
        label = self.labels[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
          image = self.transform(image).to(device)

        return image, label

Create Dataset Objects


In [ ]:
train_dataset = CustomImageDataset(dataframe=train , transform=data_transforms)
val_dataset = CustomImageDataset(dataframe=val , transform=data_transforms)
test_dataset = CustomImageDataset(dataframe=test , transform=data_transforms)

DataLoaders

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=16, shuffle=True)

In [ ]:
import torch
import torch.nn as nn

class Net(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        #Convolution Layers
        # Block 1
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)

        # Block 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)

        # Block 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(2, 2)

        #  Flatten + Fully Connected 
        flattened_size = 128 * 16 * 16
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(flattened_size, 128)
        self.relu4 = nn.ReLU()
        self.output = nn.Linear(128, num_classes) 
    def forward(self, x):
        # Conv 
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))

        # Flatten + Dense layers
        x = self.flatten(x)
        x = self.relu4(self.linear1(x))
        x = self.output(x)  

        return x

In [ ]:
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
================================================================
            Conv2d-1         [-1, 32, 128, 128]             896
              ReLU-2         [-1, 32, 128, 128]               0
         MaxPool2d-3           [-1, 32, 64, 64]               0
            Conv2d-4           [-1, 64, 64, 64]          18,496
              ReLU-5           [-1, 64, 64, 64]               0
         MaxPool2d-6           [-1, 64, 32, 32]               0
            Conv2d-7          [-1, 128, 32, 32]          73,856
              ReLU-8          [-1, 128, 32, 32]               0
         MaxPool2d-9          [-1, 128, 16, 16]               0
          Flatten-10                [-1, 32768]               0
           Linear-11                  [-1, 128]       4,194,432
             ReLU-12                  [-1, 128]               0
           Linear-13                    [-1, 3]             387
================================================================
Total params: 4,288,067
Trainable params: 4,288,067
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.19
Forward/backward pass size (MB): 16.00
Params size (MB): 16.36
Estimated Total Size (MB): 32.55
----------------------------------------------------------------

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
Training

In [ ]:


EPOCHS = 10

for epoch in range(EPOCHS):

    # ------------------ TRAIN ------------------
    model.train()

    train_loss = 0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        # Accuracy
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_accuracy = 100 * correct / total
    avg_train_loss = train_loss / len(train_loader)

    
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_accuracy = 100 * val_correct / val_total
    avg_val_loss = val_loss / len(val_loader)

    print(
        f"Epoch {epoch+1}/{EPOCHS}, "
        f"Train Loss: {avg_train_loss:.4f}, "
        f"Train Accuracy: {train_accuracy:.4f}, "
        f"Validation Loss: {avg_val_loss:.4f}, "
        f"Validation Accuracy: {val_accuracy:.4f}"
    )


In [ ]:
Epoch 1/10, Train Loss: 0.4479, Train Accuracy: 80.9406, Validation Loss: 0.2578, Validation Accuracy: 90.7025
Epoch 2/10, Train Loss: 0.1800, Train Accuracy: 93.4018, Validation Loss: 0.1544, Validation Accuracy: 94.4628
Epoch 3/10, Train Loss: 0.1038, Train Accuracy: 96.2271, Validation Loss: 0.1105, Validation Accuracy: 95.9917
Epoch 4/10, Train Loss: 0.0701, Train Accuracy: 97.3696, Validation Loss: 0.1644, Validation Accuracy: 94.0496
Epoch 5/10, Train Loss: 0.0439, Train Accuracy: 98.3704, Validation Loss: 0.1609, Validation Accuracy: 94.3802
Epoch 6/10, Train Loss: 0.0283, Train Accuracy: 98.9195, Validation Loss: 0.2031, Validation Accuracy: 95.0000
Epoch 7/10, Train Loss: 0.0295, Train Accuracy: 99.0081, Validation Loss: 0.1887, Validation Accuracy: 94.5455
Epoch 8/10, Train Loss: 0.0212, Train Accuracy: 99.2118, Validation Loss: 0.1585, Validation Accuracy: 95.9504
Epoch 9/10, Train Loss: 0.0197, Train Accuracy: 99.2915, Validation Loss: 0.3911, Validation Accuracy: 92.6860
Epoch 10/10, Train Loss: 0.0236, Train Accuracy: 99.2295, Validation Loss: 0.1630, Validation Accuracy: 95.8264